# PQ ADBC Advisor - Quickstart

Run this notebook inside your Fabric workspace to (1) discover artifacts affected by the ODBC to ADBC connector migration and (2) validate refresh success after you flip the ADBC switch.

This is a **read-only** tool. It does not rewrite any M code or change any connections.

In [ ]:
%pip install git+https://github.com/microsoft/pq-adbc-advisor.git

## Phase 1 - discovery

Run this BEFORE you enable ADBC at the tenant or workspace level. It builds an inventory of every semantic model, dataset, and dataflow in the current workspace that references one of the seven migrating connectors, and flags any that explicitly pin `Implementation="1.0"`.

In [ ]:
from pq_adbc_advisor import scan_workspace

baseline = scan_workspace()
baseline.summary()

In [ ]:
df = baseline.to_dataframe()
display(df)

In [ ]:
baseline.to_html('/lakehouse/default/Files/adbc_impact.html')

## Phase 2 - validation (after the ADBC switch)

Now enable the tenant setting **"Use ADBC drivers for supported connectors"** (or set the workspace override, or remove `Implementation="1.0"` pins).

Then run the cell below. For each impacted semantic model we trigger a fresh refresh, poll for completion, and compare status + duration to the pre-migration baseline.

In [ ]:
from pq_adbc_advisor import validate_migration

result = validate_migration(baseline)
result.summary()

In [ ]:
display(result.to_dataframe())
result.to_html('/lakehouse/default/Files/adbc_validation.html')

## Optional - tenant-wide scan

Requires Fabric admin (or an SP in the correct security group). Uses the Power BI admin Scanner API rather than per-workspace `getDefinition`.

In [ ]:
# from pq_adbc_advisor import scan_tenant
# tenant_report = scan_tenant()
# tenant_report.summary()
# tenant_report.to_html('/lakehouse/default/Files/adbc_tenant_impact.html')